In [1]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import torch
import seisbench.models as sbm
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from obspy import Stream
from glob import glob
import datetime

In [2]:
import sys
lib_path = [r'C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisRoutine',
            r'C:\Users\ikahb\OneDrive\Applications\GitHub\SeisRoutine',
            '/home/ikahbasi/Works/SeisRoutine/']
for path in lib_path:
    sys.path.append(path)
##########################################################################
import SeisRoutine.plot as srp
import SeisRoutine.catalog as src
import SeisRoutine.config as srconf

In [3]:
# logging.getLogger().setLevel(logging.INFO)
srconf.configure_logging(level='info', mode='file', filename='now')

Logging Starts in:
./_2025-12-24_20-26-00.log


In [4]:
client = Client(base_url="http://192.168.205.13:8080/")

In [5]:
dl_pickers = {
    #'PhaseNet_stead': sbm.PhaseNet.from_pretrained("stead"),
    'PhaseNet_original': sbm.PhaseNet.from_pretrained("original"),
    #   'PhaseNet_scedc': sbm.PhaseNet.from_pretrained("scedc"),
    #   'PhaseNet_instance': sbm.PhaseNet.from_pretrained("instance"),
    ############################################################################
    # 'EQTransformer_stead': sbm.EQTransformer.from_pretrained("stead"),
      'EQTransformer_original': sbm.EQTransformer.from_pretrained("original"),
    #   'EQTransformer_scedc': sbm.EQTransformer.from_pretrained("scedc"),
    #   'EQTransformer_instance': sbm.EQTransformer.from_pretrained("instance"),
    ############################################################################
    #   sbm.GPD.from_pretrained("stead"),
    # 'GPD_original': sbm.GPD.from_pretrained("original"),
    #   'GPD_scedc': sbm.GPD.from_pretrained("scedc"),
    #   'GPD_instance': sbm.GPD.from_pretrained("instance"),
}

if torch.cuda.is_available():
    for key, dl_picker in dl_pickers.items():
        dl_picker.cuda();
        logging.info(f"{key} Running on GPU")
        logging.info(dl_picker._annotate_args.get("*_threshold"))
else:
    logging.info("Running on CPU")

In [6]:
# starttime = UTCDateTime("2000-01-01")
# endtime = UTCDateTime("2010-01-01")
# cat = client.get_events(starttime=starttime,
#                         endtime=endtime,
#                         includearrivals=True)

In [7]:
results_dict = {'id': [],
                'catalog': []}
for key, dl_picker in dl_pickers.items():
    results_dict[key] = []
results_df = pd.DataFrame(results_dict)

In [14]:
for year in range(2013, 2026):
    starttime = UTCDateTime(f"{year}-01-01")
    endtime = UTCDateTime(f"{year+1}-01-01")
    msg = f'Getting Catalog from {starttime} to {endtime}'
    logging.info(msg)
    cat = client.get_events(starttime=starttime,
                            endtime=endtime,
                            includearrivals=True)
    logging.info('\n'+cat.__str__()+'\n')
    len_cat = len(cat)
    for index_ev, ev in enumerate(cat):
        precentage = index_ev / len_cat * 100
        origin = ev.preferred_origin()
        otime = origin.time
        msg = f'({index_ev} of {len_cat} - {precentage:.2f} %) {otime}'
        # if index_ev < 477:
        #     continue
        logging.info(msg)
        try:
            st = client.get_waveforms(network="*",
                                      station="*",
                                      location="*",
                                      channel="*",
                                      starttime=otime,
                                      endtime=otime + 30*60)
        except Exception as error:
            logging.warning(error)
            continue
        ## preprocessing
        st.merge(-1)
        st.detrend('constant')
        st.resample(100)
        st.merge(fill_value=0)
        filtered_stream = Stream(traces=[tr for tr in st
                                        if tr.stats.npts >= 100*30])
        ## catalog
        picks_cat_classified = {}
        dists_cat_classified = {}
        for pick in ev.picks:
            arrival = src.select_arrival_related_to_the_pick(pick, origin.arrivals)
            key = f'{pick.waveform_id.network_code}.{pick.waveform_id.station_code}._{pick.phase_hint[0].upper()}'
            if key not in picks_cat_classified.keys():
                picks_cat_classified[key] = np.array([pick.time.timestamp])
                if arrival.distance:
                    dists_cat_classified[key] = np.array([arrival.distance*111])
                else:
                    dists_cat_classified[key] = np.array([np.nan])
            else:
                picks_cat_classified[key] = np.append(picks_cat_classified[key],
                                                        pick.time.timestamp)
                if arrival.distance:
                    dists_cat_classified[key] = np.append(dists_cat_classified[key],
                                                          arrival.distance*111)
                else:
                    dists_cat_classified[key] = np.append(dists_cat_classified[key],
                                                          np.nan)
            ###
        series = [pd.Series(picks_cat_classified, name='catalog_picktime'),
                pd.Series(dists_cat_classified, name='catalog_pickdist')]
        ## picker
        for picker_name, picker_model in dl_pickers.items():
            # print(picker_name)
            outputs = picker_model.classify(st)
            picks = outputs.picks
            stations = {pick.trace_id for pick in picks}
            stations = [station.split('.')[1] for station in stations]
            picks_DL_classified = {}
            for pick in picks:
                key = f'{pick.trace_id}_{pick.phase}'
                if key not in picks_DL_classified.keys():
                    picks_DL_classified[key] = np.array([pick.peak_time.timestamp])
                else:
                    picks_DL_classified[key] = np.append(picks_DL_classified[key],
                                                        pick.peak_time.timestamp)
            series.append(pd.Series(picks_DL_classified, name=f'{picker_name}_picktime'))
        ##
        df_ev = pd.concat(series, axis=1)
        df_ev['otime'] = origin.time
        df_ev['latitude'] = origin.latitude
        df_ev['longitude'] = origin.longitude
        df_ev['depth'] = origin.depth
        df_ev['resource_id'] = origin.resource_id
        ###
        outpath = '/home/ikahbasi/Documents/Results'
        name = otime.strftime('%Y-%m-%dT%H-%M-%S.%f') + '.pkl'
        df_ev.to_pickle(f'{outpath}/{name}')

/home/ikahbasi/Applications/miniconda3/envs/seisbench_gpu/lib/python3.13/site-packages/obspy/core/stream.py:3043: UserWarning: Incompatible traces (sampling_rate, dtype, ...) with same id detected. Doing nothing.
  warnings.warn(msg)
/home/ikahbasi/Applications/miniconda3/envs/seisbench_gpu/lib/python3.13/site-packages/obspy/core/stream.py:3043: UserWarning: Incompatible traces (sampling_rate, dtype, ...) with same id detected. Doing nothing.
  warnings.warn(msg)
/home/ikahbasi/Applications/miniconda3/envs/seisbench_gpu/lib/python3.13/site-packages/obspy/core/stream.py:3043: UserWarning: Incompatible traces (sampling_rate, dtype, ...) with same id detected. Doing nothing.
  warnings.warn(msg)
/home/ikahbasi/Applications/miniconda3/envs/seisbench_gpu/lib/python3.13/site-packages/obspy/core/stream.py:3043: UserWarning: Incompatible traces (sampling_rate, dtype, ...) with same id detected. Doing nothing.
  warnings.warn(msg)
/home/ikahbasi/Applications/miniconda3/envs/seisbench_gpu/lib/py

In [ ]:
from glob import glob

In [ ]:
lst_df_ev = []

outpath = '/home/ikahbasi/Documents/Results'
for fpath in glob(f'{outpath}/*.pkl'):
    df_ev = pd.read_pickle(fpath)
    lst_df_ev.append(df_ev)
df_cat = pd.concat(lst_df_ev, axis=0)

In [ ]:
lst = []
for key, val in df_cat['catalog_picktime'].items():
    # print(key, val)
    if isinstance(val, np.ndarray):
        if len(val) == 1:
            lst.append(val[0])
        else:
            if key.endswith('_P'):
                lst.append(min(val))
            elif key.endswith('_S'):
                lst.append(np.mean(val))
    elif np.isnan(val):
        lst.append(np.nan)
    else:
        raise ValueError
df_cat['catalog_picktime'] = lst

In [ ]:
lst = []
for key, val in df_cat['catalog_pickdist'].items():
    # print(key, val)
    if isinstance(val, np.ndarray):
        if len(val) == 1:
            lst.append(val[0])
        else:
            if key.endswith('_P'):
                lst.append(min(val))
            elif key.endswith('_S'):
                lst.append(np.mean(val))
    elif np.isnan(val):
        lst.append(np.nan)
    else:
        raise ValueError
df_cat['catalog_pickdist'] = lst

In [ ]:
keys = [key for key in df_cat.keys() if key.endswith('_picktime')]
diff = df_cat[keys].apply(lambda x: x-x['catalog_picktime'], axis=1)
diff['catalog_pickdist'] = df_cat['catalog_pickdist']

In [ ]:
# df_cat[keys].apply(lambda x: x-x['catalog_picktime'])#  - df_cat['catalog_picktime'].values
# # df_cat['catalog_picktime'] - df_cat['catalog_picktime']

In [ ]:
# keys = [key for key in df_cat.keys() if key.endswith('_picktime')]
# df_selected = df_cat[keys] - df_cat['catalog_picktime']
# df_selected

In [ ]:
# keys = [key for key in df_cat.keys() if key.endswith('_picktime')]
# df_selected = df_cat[keys]
# diff = df_selected['PhaseNet_original_picktime'] - df_selected['catalog_picktime']
# np.concat(diff.values)

In [ ]:
# srp.histogram(arr2, orientation='vertical', figsize=(6, 4))
# srp.histogram(arr2, orientation='vertical', figsize=(6, 4), title=f'{key}  {phase}')

In [ ]:
# non_empty_arrays = [arr for arr in filtered[key].values if isinstance(arr, np.ndarray) and arr.size > 0]
# non_empty_arrays

In [ ]:
# [arr for arr in filtered['catalog_pickdist'].values if isinstance(arr, np.ndarray) and arr.size > 0]

In [ ]:
diff

In [ ]:
diff['id'] = diff.index
diff.reset_index(drop=True)
keys = diff.keys().tolist()
keys.remove('catalog_picktime')
keys.remove('catalog_pickdist')
keys.remove('id')

# diff_filtered = diff[diff['catalog_picktime'].notna()]
for phase in ['_P', '_S']:
    diff_filtered = diff[diff['id'].str.endswith(phase)]
    print(phase)
    # print(filtered)
    for key in keys:
        diff_filtered2 = diff_filtered.explode(key)
        diff_filtered2 = diff_filtered2[diff_filtered2[key].notna()]

        # diff_filtered2[key, 'catalog_pickdist']
        arr_error = diff_filtered2[key].values
        arr_dist = diff_filtered2['catalog_pickdist'].values
        msk = np.abs(arr_error) <= 10
        arr_error = arr_error[msk].tolist()
        arr_dist = arr_dist[msk].tolist()
        #
        # arr_dist = filtered['catalog_pickdist'][msk].values
        srp.histogram(arr_error,
                      orientation='vertical',
                      figsize=(6, 4), title=f'{key}  {phase}')
        srp.density_hist(x=arr_dist, y=arr_error,
                         xstep=5, ystep=0.25,
                         ylim=[-4, 4],
                         figsize=(10, 4), title=f'{key}  {phase}')

In [ ]:
def extract_first_element(x):
    if isinstance(x, list):
        return x[0] if len(x) > 0 else np.nan
    elif isinstance(x, np.ndarray):
        return x[0] if x.size > 0 else np.nan
    else:
        return x

In [ ]:
for col in keys:
    df_cat[col] = df_cat[col].apply(extract_first_element)

In [ ]:
df_cat.count()

In [ ]:
for col in keys:
    mask = df_cat['catalog_picktime'].notna() & df_cat[col].isna()
    count = mask.sum()
    print(f"catalog [Detected], {col} [Not Detected] {count}")
    ##
    mask = df_cat['catalog_picktime'].isna() & df_cat[col].notna()
    count = mask.sum()
    print(f"catalog [Not Detected], {col} [Detected] {count}")
    ##
    mask = df_cat['catalog_picktime'].notna() & df_cat[col].notna()
    count = mask.sum()
    print(f"catalog [Detected], {col} [Detected] {count}")
    ##
    target = df_cat[mask]
    

In [ ]:
for col in keys:
    mask = df_cat['catalog_picktime'].notna() & df_cat[col].notna()
    count = mask.sum()
    print(f"catalog [Detected], {col} [Detected] {count}")